# Exploratory Data Analysis (EDA) - Walmart Store Sales

Este notebook tiene como objetivo inicializar el entorno, consolidar la base de datos cruda (`train`, `stores`, `features`) y establecer la métrica principal de la competencia (WMAE).

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 1. Definir rutas a los datos
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

## 1. Carga y Consolidación de Datos
Uniremos los datos de ventas semanales (`train.csv`), la metadata estática de las tiendas (`stores.csv`) y las covariables temporales (`features.csv`). Es crítico hacer los *merges* correctamente usando `Store` y `Date` para evitar duplicados.

In [ ]:
def load_and_merge_data() -> pd.DataFrame:
    # Carga los CSVs crudos y consolida en un solo DataFrame
    train = pd.read_csv(os.path.join(RAW_DIR, "train.csv"))
    stores = pd.read_csv(os.path.join(RAW_DIR, "stores.csv"))
    features = pd.read_csv(os.path.join(RAW_DIR, "features.csv"))
    
    # Casteo a datetime
    train['Date'] = pd.to_datetime(train['Date'])
    features['Date'] = pd.to_datetime(features['Date'])
    
    # Merge train y stores
    merged = train.merge(stores, on='Store', how='left')
    # Merge con features (Store + Date + IsHoliday)
    merged = merged.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
    
    return merged.sort_values(by=['Store', 'Dept', 'Date']).reset_index(drop=True)

# Cargar datos
df_train = load_and_merge_data()
df_train.head()

## 2. Análisis de Valores Nulos (NaN)
Sabemos de antemano que las variables correspondientes a promociones (`MarkDown1` al `MarkDown5`) solo empezaron a registrarse de manera formal desde Noviembre de 2011. Por lo tanto, esperamos una tasa muy alta de valores nulos durante la mayor parte del conjunto de *train*. Evaluamos su dispersión.

In [ ]:
# Chequeo de NaNs
nan_summary = df_train.isnull().mean() * 100
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)
display(nan_summary)

## 3. Métrica de Evaluación: WMAE
Kaggle evalúa esta competencia estrictamente con el **Weighted Mean Absolute Error (WMAE)**.
La particularidad de esta métrica es que **los errores cometidos durante semanas feriadas (Super Bowl, Labor Day, Thanksgiving, Christmas) pesan 5 veces más** que una semana estándar. Recrearemos esta función para utilizarla en nuestra validación posterior.

In [ ]:
def calculate_wmae(y_true: pd.Series, y_pred: pd.Series, is_holiday: pd.Series) -> float:
    """
    Calcula el Weighted Mean Absolute Error (WMAE).
    Las semanas con IsHoliday=True tienen un peso de 5, el resto de 1.
    """
    weights = np.where(is_holiday, 5, 1)
    wmae = np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)
    return wmae

# Prueba de sanidad de la métrica
dummy_pred = pd.Series(np.full_like(df_train['Weekly_Sales'], df_train['Weekly_Sales'].mean()))
wmae_dummy = calculate_wmae(df_train['Weekly_Sales'], dummy_pred, df_train['IsHoliday'])
print(f"WMAE de modelo dummy (promedio global): {wmae_dummy:.2f}")